In [ ]:
project_path = "/home/jupyter"
import os
import sys

sys.path.append(project_path)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
from google.cloud import bigquery

from fintrans_toolbox.src import bq_utils as bq
from fintrans_toolbox.src import table_utils as t


client = bigquery.Client()


In [ ]:
# Summarise the data by UK Cardholder Domestic Spending All Quarterly --------------- Cardholders' Number Total Quarterly 

UK_spending_by_Dom_All1 = '''SELECT time_period_value, cardholders, destination_country, spend 
FROM `ons-fintrans-data-prod.fintrans_visa.spend_origin_and_channel` 
where time_period = 'Quarter' 
and mcc = 'All' 
and mcg = 'All' 
and merchant_channel = 'All' 
and cardholder_origin_country = 'All' 
and cardholder_origin = 'UNITED KINGDOM' 
and destination_country = 'UNITED KINGDOM' 
GROUP BY cardholders, destination_country, 
time_period_value, spend 
ORDER BY time_period_value, destination_country DESC'''
df_by_Dom_All = bq.read_bq_table_sql(client, UK_spending_by_Dom_All1)
df_by_Dom_All.head()

# Caculate UK Domestic Total Spending Quarterly

# Assuming df_by_Dom_All is the DataFrame returned from the BigQuery query
# Then group by 'time_period_value' and sum the 'spend' for each quarter

# Check if df_by_Dom_All is not None and has the expected columns
if df_by_Dom_All is not None and 'time_period_value' in df_by_Dom_All.columns and 'spend' in df_by_Dom_All.columns:
    # Group by quarter and sum the spend
    UK_spending_by_Dom_All1 = df_by_Dom_All.groupby('time_period_value')['cardholders'].sum().reset_index()
   
 # Rename the column
    UK_spending_by_Dom_All1 = UK_spending_by_Dom_All1.rename(columns={'cardholders': 'Dom_spend_All_cardholders'})
    print(UK_spending_by_Dom_All1)
else:
    print("DataFrame is empty or missing required columns.")
    
# Save the result to a CSV file
csv_filename = "UK_spending_by_Dom_All1.csv"
UK_spending_by_Dom_All1.to_csv(csv_filename, index=False)

print(f"CSV file '{csv_filename}' has been created successfully.")


In [ ]:
# Summarise the data by UK Cardholder Abroad Spending All Quarterly --------------- Cardholders' Number Total Quarterly 

UK_spending_by_Intl_All1 = '''SELECT time_period_value, cardholders, destination_country, spend 
FROM `ons-fintrans-data-prod.fintrans_visa.spend_origin_and_channel` 
where time_period = 'Quarter' 
and mcc = 'All' 
and mcg = 'All' 
and merchant_channel = 'All' 
and cardholder_origin_country = 'All' 
and cardholder_origin = 'UNITED KINGDOM' 
and destination_country != 'UNITED KINGDOM' 
GROUP BY cardholders, destination_country, 
time_period_value, spend 
ORDER BY time_period_value, destination_country DESC'''
df_by_Intl_All = bq.read_bq_table_sql(client, UK_spending_by_Intl_All1)
df_by_Intl_All.head()

# Caculate UK Abroad Total Spending Quarterly

# Assuming df_by_Intl_All is the DataFrame returned from the BigQuery query
# Then group by 'time_period_value' and sum the 'spend' for each quarter

# Check if df_by_Dom_All is not None and has the expected columns
if df_by_Intl_All is not None and 'time_period_value' in df_by_Intl_All.columns and 'spend' in df_by_Intl_All.columns:
    # Group by quarter and sum the spend
    UK_spending_by_Intl_All1 = df_by_Intl_All.groupby('time_period_value')['cardholders'].sum().reset_index()
   
 # Rename the column
    UK_spending_by_Intl_All1 = UK_spending_by_Intl_All1.rename(columns={'cardholders': 'Intl_spend_All_cardholders'})
    print(UK_spending_by_Intl_All1)
else:
    print("DataFrame is empty or missing required columns.")
    
    # Save the result to a CSV file
csv_filename = "UK_spending_by_Intl_All1.csv"
UK_spending_by_Intl_All1.to_csv(csv_filename, index=False)

print(f"CSV file '{csv_filename}' has been created successfully.")


In [ ]:
# Formula : Adjusted Quarterly Spend=( Dom_spend_All_cardholders (quarter) / Dom_spend_All_cardholders (2019Q1)) × Dom_spend_All (2019Q1)

import pandas as pd

# Assuming df_by_Dom_All is already loaded and contains the necessary columns
# Filter for 2019Q1 cardholder count
cardholders_2019Q1_row = df_by_Dom_All[df_by_Dom_All["time_period_value"] == "2019Q1"]

if not cardholders_2019Q1_row.empty:
    cardholders_2019Q1 = cardholders_2019Q1_row["cardholders"].values[0]

    # Calculate adjusted spend using 2019Q1 cardholder base
    df_by_Dom_All["adjusted_spend_2019Q1_base"] = (
        (cardholders_2019Q1 / df_by_Dom_All["cardholders"]) * df_by_Dom_All["spend"]
    )

    # Display the updated DataFrame
    print(df_by_Dom_All.head())
else:
    print("2019Q1 data not found in the dataset.")



In [ ]:
# UK Domestic Total Adjusted base value 2019Q1

import pandas as pd

# Assuming df_by_Dom_All is already loaded and contains the necessary columns
# Filter for 2019Q1 cardholder count
cardholders_2019Q1_row = df_by_Dom_All[df_by_Dom_All["time_period_value"] == "2019Q1"]

if not cardholders_2019Q1_row.empty:
    cardholders_2019Q1 = cardholders_2019Q1_row["cardholders"].values[0]

    # Calculate adjusted spend using 2019Q1 cardholder base
    df_by_Dom_All["adjusted_spend_2019Q1_base"] = (
        (cardholders_2019Q1 / df_by_Dom_All["cardholders"]) * df_by_Dom_All["spend"]
    )

    # Save the updated DataFrame to a CSV file
    df_by_Dom_All.to_csv("adjusted_spend_2019Q1_base_Dom.csv", index=False)
    print("Data saved to adjusted_spend_2019Q1_base_Dom.csv")
else:
    print("2019Q1 data not found in the dataset.")



In [ ]:
# Summarise the data by UK Cardholder Domestic Online Spending All Quarterly --------------- Cardholders' Number Total Quarterly 

UK_spending_by_Dom_Online = '''SELECT time_period_value, cardholders, destination_country, spend 
FROM `ons-fintrans-data-prod.fintrans_visa.spend_origin_and_channel` 
where time_period = 'Quarter' 
and mcc = 'All' 
and mcg = 'All' 
and merchant_channel = 'Online' 
and cardholder_origin_country = 'All' 
and cardholder_origin = 'UNITED KINGDOM' 
and destination_country = 'UNITED KINGDOM' 
GROUP BY cardholders, destination_country, 
time_period_value, spend 
ORDER BY time_period_value, destination_country DESC'''
df_by_Dom_Online = bq.read_bq_table_sql(client, UK_spending_by_Dom_Online)
df_by_Dom_Online.head()

# Caculate UK Domestic Online Total Spending Quarterly

# Assuming df_by_Dom_Online is the DataFrame returned from the BigQuery query
# Then group by 'time_period_value' and sum the 'spend' for each quarter

# Check if df_by_Dom_Online is not None and has the expected columns
if df_by_Dom_Online is not None and 'time_period_value' in df_by_Dom_Online.columns and 'spend' in df_by_Dom_Online.columns:
    # Group by quarter and sum the spend
    UK_spending_by_Dom_Online = df_by_Dom_Online.groupby('time_period_value')['cardholders'].sum().reset_index()
   
 # Rename the column
    UK_spending_by_Dom_Online = UK_spending_by_Dom_Online.rename(columns={'cardholders': 'Dom_spend_Online_cardholders'})
    print(UK_spending_by_Dom_Online)
else:
    print("DataFrame is empty or missing required columns.")
    
# Save the result to a CSV file
csv_filename = "UK_spending_by_Dom_Online.csv"
UK_spending_by_Dom_Online.to_csv(csv_filename, index=False)

print(f"CSV file '{csv_filename}' has been created successfully.")


In [ ]:
# UK Domestic Online Total Adjusted base value 2019Q1


# Summarise the data by UK Cardholder Domestic Online Spending All

UK_spending_by_Dom_Online_All = '''SELECT time_period_value, destination_country, spend 
FROM `ons-fintrans-data-prod.fintrans_visa.spend_origin_and_channel` 
where time_period = 'Quarter' 
and mcg = 'All' 
and merchant_channel = 'Online' 
and cardholder_origin_country = 'All' 
and cardholder_origin = 'UNITED KINGDOM' 
and destination_country = 'UNITED KINGDOM' 
GROUP BY destination_country, 
time_period_value, spend 
ORDER BY time_period_value, destination_country DESC'''
df_by_Dom_Online_All = bq.read_bq_table_sql(client, UK_spending_by_Dom_Online_All)

df_by_Dom_Online_All.head()

# Sum UK Cardholder Domestic Online Spending All Quarterly
# Assuming df_by_Dom_Online_All is the DataFrame returned from the BigQuery query
# Then group by 'time_period_value' and sum the 'spend' for each quarter

# Check if df_by_Dom_Online_All is not None and has the expected columns
if df_by_Dom_Online_All is not None and 'time_period_value' in df_by_Dom_Online_All.columns and 'spend' in df_by_Dom_Online_All.columns:
    # Group by quarter and sum the spend
    UK_spending_by_Dom_Online_All = df_by_Dom_Online_All.groupby('time_period_value')['spend'].sum().reset_index()
   
 # Rename the column
    UK_spending_by_Dom_Online_All = UK_spending_by_Dom_Online_All.rename(columns={'spend': 'Dom_spend_Online_All'})
    print(UK_spending_by_Dom_Online_All)
else:
    print("DataFrame is empty or missing required columns.")
    
    # Save the result to a CSV file
csv_filename = "UK_spending_by_Dom_Online_All.csv"
UK_spending_by_Dom_Online_All.to_csv(csv_filename, index=False)

print(f"CSV file '{csv_filename}' has been created successfully.")



In [ ]:
# UK Domestic Online Total Adjusted base value 2019Q1

import pandas as pd

# Load the data from the two CSV files
df_cardholders = pd.read_csv("UK_spending_by_Dom_Online.csv")
df_spend = pd.read_csv("UK_spending_by_Dom_Online_All.csv")

# Merge the two DataFrames on 'time_period_value'
df_merged = pd.merge(df_cardholders, df_spend, on="time_period_value", how="inner")

# Extract the 2019Q1 values
base_row = df_merged[df_merged["time_period_value"] == "2019Q1"]
if not base_row.empty:
    base_cardholders = base_row["Dom_spend_Online_cardholders"].values[0]
    base_spend = base_row["Dom_spend_Online_All"].values[0]

    # Calculate adjusted quarterly spend
    df_merged["adjusted_Online_Dom_spend"] = (
        df_merged["Dom_spend_Online_cardholders"] / base_cardholders
    ) * base_spend

    # Save the result to a new CSV file
    df_merged.to_csv("Adjusted_Online_Dom_Spend.csv", index=False)
    print("Adjusted quarterly spend saved to Adjusted_Online_Dom_Spend.csv")
else:
    print("2019Q1 base data not found in the dataset.")



In [ ]:
# Summarise the data by UK Cardholder Domestic Face to Face Spending All Quarterly --------------- Cardholders' Number Total Quarterly 

UK_spending_by_Dom_F2F = '''SELECT time_period_value, cardholders, destination_country, spend 
FROM `ons-fintrans-data-prod.fintrans_visa.spend_origin_and_channel` 
where time_period = 'Quarter' 
and mcc = 'All' 
and mcg = 'All' 
and merchant_channel = 'Face to Face' 
and cardholder_origin_country = 'All' 
and cardholder_origin = 'UNITED KINGDOM' 
and destination_country = 'UNITED KINGDOM' 
GROUP BY cardholders, destination_country, 
time_period_value, spend 
ORDER BY time_period_value, destination_country DESC'''
df_by_Dom_F2F = bq.read_bq_table_sql(client, UK_spending_by_Dom_F2F)
df_by_Dom_F2F.head()

# Caculate UK Domestic Face to Face Total Spending Quarterly

# Assuming df_by_Dom_F2F is the DataFrame returned from the BigQuery query
# Then group by 'time_period_value' and sum the 'spend' for each quarter

# Check if df_by_Dom_F2F is not None and has the expected columns
if df_by_Dom_F2F is not None and 'time_period_value' in df_by_Dom_F2F.columns and 'spend' in df_by_Dom_F2F.columns:
    # Group by quarter and sum the spend
    UK_spending_by_Dom_F2F = df_by_Dom_F2F.groupby('time_period_value')['cardholders'].sum().reset_index()
   
 # Rename the column
    UK_spending_by_Dom_F2F = UK_spending_by_Dom_F2F.rename(columns={'cardholders': 'Dom_spend_F2F_cardholders'})
    print(UK_spending_by_Dom_F2F)
else:
    print("DataFrame is empty or missing required columns.")
    
# Save the result to a CSV file
csv_filename = "UK_spending_by_Dom_F2F.csv"
UK_spending_by_Dom_F2F.to_csv(csv_filename, index=False)

print(f"CSV file '{csv_filename}' has been created successfully.")

In [ ]:
# UK Domestic Face to Face Total Adjusted base value 2019Q1


# Summarise the data by UK Cardholder Domestic Face to Face Spending All

UK_spending_by_Dom_F2F_All = '''SELECT time_period_value, destination_country, spend 
FROM `ons-fintrans-data-prod.fintrans_visa.spend_origin_and_channel` 
where time_period = 'Quarter' 
and mcg = 'All' 
and merchant_channel = 'Face to Face' 
and cardholder_origin_country = 'All' 
and cardholder_origin = 'UNITED KINGDOM' 
and destination_country = 'UNITED KINGDOM' 
GROUP BY destination_country, 
time_period_value, spend 
ORDER BY time_period_value, destination_country DESC'''
df_by_Dom_F2F_All = bq.read_bq_table_sql(client, UK_spending_by_Dom_F2F_All)

df_by_Dom_F2F_All.head()

# Sum UK Cardholder Domestic Face to Face Spending All Quarterly
# Assuming df_by_Dom_F2F_All is the DataFrame returned from the BigQuery query
# Then group by 'time_period_value' and sum the 'spend' for each quarter

# Check if df_by_Dom_F2F_All is not None and has the expected columns
if df_by_Dom_F2F_All is not None and 'time_period_value' in df_by_Dom_F2F_All.columns and 'spend' in df_by_Dom_F2F_All.columns:
    # Group by quarter and sum the spend
    UK_spending_by_Dom_F2F_All = df_by_Dom_F2F_All.groupby('time_period_value')['spend'].sum().reset_index()
   
 # Rename the column
    UK_spending_by_Dom_F2F_All = UK_spending_by_Dom_F2F_All.rename(columns={'spend': 'Dom_spend_F2F_All'})
    print(UK_spending_by_Dom_F2F_All)
else:
    print("DataFrame is empty or missing required columns.")
    
    # Save the result to a CSV file
csv_filename = "UK_spending_by_Dom_F2F_All.csv"
UK_spending_by_Dom_F2F_All.to_csv(csv_filename, index=False)

print(f"CSV file '{csv_filename}' has been created successfully.")



In [ ]:
# UK Domestic Face to Face Total Adjusted base value 2019Q1

import pandas as pd

# Load the data from the two CSV files
df_cardholders = pd.read_csv("UK_spending_by_Dom_F2F.csv")
df_spend = pd.read_csv("UK_spending_by_Dom_F2F_All.csv")

# Merge the two DataFrames on 'time_period_value'
df_merged = pd.merge(df_cardholders, df_spend, on="time_period_value", how="inner")

# Extract the 2019Q1 values
base_row = df_merged[df_merged["time_period_value"] == "2019Q1"]
if not base_row.empty:
    base_cardholders = base_row["Dom_spend_F2F_cardholders"].values[0]
    base_spend = base_row["Dom_spend_F2F_All"].values[0]

    # Calculate adjusted quarterly spend
    df_merged["adjusted_F2F_Dom_spend"] = (
        df_merged["Dom_spend_F2F_cardholders"] / base_cardholders
    ) * base_spend

    # Save the result to a new CSV file
    df_merged.to_csv("Adjusted_F2F_Dom_Spend.csv", index=False)
    print("Adjusted quarterly spend saved to Adjusted_F2F_Dom_Spend.csv")
else:
    print("2019Q1 base data not found in the dataset.")

In [ ]:
# Adjusted Indexed Spend Dom Online & F2F
# Indexed card spending data (average 2019 equals 100) is calculated :
# Spend=(Adjusted Spend / Average Adjusted Spend in 2019) × 100

import pandas as pd

# Define a function to calculate Index Adjusted Spend
def calculate_index_adjusted_spend(df, spend_column, year_column='time_period_value', base_year=2019):
    # Filter the data for the base year
    base_year_data = df[df[year_column] == base_year]
    
    # Calculate the average adjusted spend for the base year
    average_base_year_spend = base_year_data[spend_column].mean()
    
    # Calculate the index adjusted spend
    df['Index_Adjusted_Spend'] = (df[spend_column] / average_base_year_spend) * 100
    
    return df

# Load the CSV files
f2f_df = pd.read_csv('Adjusted_F2F_Dom_Spend.csv')
online_df = pd.read_csv('Adjusted_Online_Dom_Spend.csv')

# Apply the function to both datasets
f2f_indexed = calculate_index_adjusted_spend(f2f_df, spend_column='adjusted_F2F_Dom_spend')
online_indexed = calculate_index_adjusted_spend(online_df, spend_column='adjusted_Online_Dom_spend')

# Save the results to new CSV files
f2f_indexed.to_csv('indexed_adjusted_F2F_Dom_spend.csv', index=False)
online_indexed.to_csv('indexed_adjusted_Online_Dom_spend.csv', index=False)

print("Indexed CSV files have been created successfully.")




In [ ]:
# Line chart for Dom Online & F2F Adjusted Indexed Value

import pandas as pd
import matplotlib.pyplot as plt

# Load the indexed CSV files
f2f_file = "indexed_adjusted_F2F_Dom_spend.csv"
online_file = "indexed_Adjust_Quarterly_Dom_Spend_Online.csv"

# Read the data
f2f_df = pd.read_csv(f2f_file)
online_df = pd.read_csv(online_file)

# Sort by time_period_value
f2f_df = f2f_df.sort_values("time_period_value")
online_df = online_df.sort_values("time_period_value")

# Merge the two datasets on time_period_value
merged_df = pd.merge(f2f_df, online_df, on="time_period_value", how="inner")

# Plot the indexed trends
plt.figure(figsize=(12, 6))
plt.plot(merged_df["time_period_value"], merged_df["adjusted_F2F_Dom_spend"], label="F2F Adjusted Spend Index", marker='o')
plt.plot(merged_df["time_period_value"], merged_df["adjusted_Online_Dom_spend"], label="Online Adjusted Spend Index", marker='o')
plt.xticks(rotation=45)
plt.xlabel("Quarter")
plt.ylabel("Index Adjusted Spend")
plt.title("Domestic Quarterly Index Adjusted Spend: F2F vs Online")
plt.legend()
plt.tight_layout()
plt.grid(True)
plt.savefig("Index_Adjusted_Spend_Comparison.png")
plt.show()

